In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ML_Rain") \
    .master("local[*]") \
    .getOrCreate()

In [2]:
data_path = "hdfs://localhost:9000/DACK/weather_ml_rain"
df = spark.read.parquet(data_path)

In [3]:
# Hiển thị thử 5 dòng để kiểm tra
df.select("scaled_features", "label").show(5, truncate=False)

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|scaled_features                                                                                                                                                                                                                                                                                                                                                                                     |label|
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [4]:
# 2. Chia tập dữ liệu: 80% để Học, 20% để Thi (Dùng seed=42 để cố định kết quả)
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)
print(f"Số lượng dòng tập Học (Train): {train_data.count()}")
print(f"Số lượng dòng tập Thi (Test): {test_data.count()}")

Số lượng dòng tập Học (Train): 113633
Số lượng dòng tập Thi (Test): 28560


In [5]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# ROC-AUC
roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

# F1-Score
f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

Logistic Regression

In [6]:
# 1. Import đúng thuật toán Logistic Regression
from pyspark.ml.classification import LogisticRegression
import time

print(">>> BẮT ĐẦU: LOGISTIC REGRESSION")
start_time = time.time()

# Bước 1: Khai báo mô hình
lr = LogisticRegression(featuresCol="scaled_features", labelCol="label")

# Bước 2: Cho mô hình HỌC trên tập Train
lr_model = lr.fit(train_data)

# Bước 3: Cho mô hình LÀM BÀI THI trên tập Test
lr_predictions = lr_model.transform(test_data)

# Bước 4: Dùng thước đã tạo ở Phần 2 để CHẤM ĐIỂM
lr_roc_auc = roc_evaluator.evaluate(lr_predictions)
lr_f1 = f1_evaluator.evaluate(lr_predictions)

# In kết quả
print(f"Hoàn thành trong: {round(time.time() - start_time, 2)} giây")
print(f"Điểm ROC-AUC : {lr_roc_auc:.4f}")
print(f"Điểm F1-Score: {lr_f1:.4f}\n")

>>> BẮT ĐẦU: LOGISTIC REGRESSION
Hoàn thành trong: 15.54 giây
Điểm ROC-AUC : 0.8517
Điểm F1-Score: 0.8269



In [7]:
# 1. Import đúng thuật toán Random Forest
from pyspark.ml.classification import RandomForestClassifier

print(">>> BẮT ĐẦU: RANDOM FOREST")
# start_time = time.time()

# Bước 1: Khai báo mô hình (Trồng 50 cây, mỗi cây sâu tối đa 5 tầng)
rf = RandomForestClassifier(featuresCol="scaled_features", labelCol="label", numTrees=50, maxDepth=5)

# Bước 2: Cho mô hình HỌC
rf_model = rf.fit(train_data)

# Bước 3: Cho mô hình LÀM BÀI THI
rf_predictions = rf_model.transform(test_data)

# Bước 4: CHẤM ĐIỂM
rf_roc_auc = roc_evaluator.evaluate(rf_predictions)
rf_f1 = f1_evaluator.evaluate(rf_predictions)

# In kết quả
# print(f"Hoàn thành trong: {round(time.time() - start_time, 2)} giây")
print(f"Điểm ROC-AUC : {rf_roc_auc:.4f}")
print(f"Điểm F1-Score: {rf_f1:.4f}\n")

>>> BẮT ĐẦU: RANDOM FOREST
Điểm ROC-AUC : 0.8285
Điểm F1-Score: 0.8124

